<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
由 <a href="https://sebastianraschka.com">Sebastian Raschka</a> 撰写的 <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> 一书配套的补充代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 附录 F：LLM 评估的常见方法

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # 供 reasoning_from_scratch 使用
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

&nbsp;
## F.1 理解 LLM 的主要评估方法

- 本节没有代码

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F01_raschka.webp" width="500px">

&nbsp;
### F.2 评估选项回答的准确率

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F02_raschka.webp" width="500px">

- 请注意，该图展示的是一种简化的多选题评估方式（如 MMLU），即将模型生成的答案字母与正确答案字母进行比对
- 实践中还有对数概率评分等变体，不再只看最终字母，而是计算模型认为每个候选答案的可能性
- 对于推理模型，这也可以包括在把正确答案输入模型时，它再次生成该答案的可能性
- 无论哪种方式，评估的核心都是检查模型是否选中了某个预先定义的答案
- （有关输出概率得分的更多内容，参见第 4 章对文本生成函数的改进）

&nbsp;
#### F.2.1 加载模型

In [ ]:
from pathlib import Path
import torch

from reasoning_from_scratch.ch02 import (
    get_device
)
from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer,
    Qwen3Model,
    QWEN_CONFIG_06_B
)

device = get_device()
torch.set_float32_matmul_precision("high")

# 如果遇到兼容性问题，请尝试
# 取消下面一行的注释，然后重新运行 notebook
# device = "cpu"

WHICH_MODEL = "base"

if WHICH_MODEL == "base":

    download_qwen3_small(
        kind="base", tokenizer_only=False, out_dir="qwen3"
    )

    tokenizer_path = Path("qwen3") / "tokenizer-base.json"
    model_path = Path("qwen3") / "qwen3-0.6B-base.pth"
    tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

elif WHICH_MODEL == "reasoning":

    download_qwen3_small(
        kind="reasoning", tokenizer_only=False, out_dir="qwen3"
    )

    tokenizer_path = Path("qwen3") / "tokenizer-reasoning.json"
    model_path = Path("qwen3") / "qwen3-0.6B-reasoning.pth"
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_path,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=True,
    )

else:
    raise ValueError(f"Invalid choice: WHICH_MODEL={WHICH_MODEL}")


model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))

model.to(device)


USE_COMPILE = False  # 设为 True 以启用编译
if USE_COMPILE:
  torch._dynamo.config.allow_unspec_int_on_nn_module = True
  model = torch.compile(model)

&nbsp;
#### F.2.2 检查生成的答案字母

In [ ]:
example = {
    "question": (
        "How many ways are there to put 4 distinguishable"
        " balls into 2 indistinguishable boxes?"
    ),
    "choices": ["7", "11", "16", "8"],
    "answer": "D",
}

def format_prompt(example):
    return (
        f"{example['question']}\n"
        f"A. {example['choices'][0]}\n"
        f"B. {example['choices'][1]}\n"
        f"C. {example['choices'][2]}\n"
        f"D. {example['choices'][3]}\n"
        "Answer: "  # 末尾空格会促使模型生成单字母的下一词元
    )

prompt = format_prompt(example)
print(prompt)

---


- 可以通过 `datasets` 库直接加载 MMLU 数据集中的示例（可用 `pip install datasets` 或 `uv add datasets` 安装）：

```python
from datasets import load_dataset

configs = get_dataset_config_names("cais/mmlu")
dataset = load_dataset("cais/mmlu", "high_school_mathematics")

# 查看测试集中的第一个样本：
example = dataset["test"][0]
print(example)
```

- 上例使用 "high_school_mathematics" 子集；若想查看其他子集列表，可运行下方代码：


```python
from datasets import get_dataset_config_names

subsets = get_dataset_config_names("cais/mmlu")
print(subsets)
```

---


In [ ]:
prompt_ids = tokenizer.encode(prompt)
prompt_fmt = torch.tensor(prompt_ids, device=device).unsqueeze(0)

- 我们生成少量 token，并提取模型首次输出的 A/B/C/D 字母：

In [ ]:
from reasoning_from_scratch.ch02 import generate_text_basic_stream_cache


def predict_choice(
    model, tokenizer, prompt_fmt, max_new_tokens=8
):
    pred = None
    for t in generate_text_basic_stream_cache(
        model=model,
        token_ids=prompt_fmt,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
    ):
        answer = tokenizer.decode(t.squeeze(0).tolist())
        for letter in answer:
            letter = letter.upper()
            if letter in "ABCD":
                pred = letter
                break
        if pred:  # 一旦出现字母就停止
            break
    return pred

In [ ]:
pred1 = predict_choice(model, tokenizer, prompt_fmt)

print(
    f"Generated letter: {pred1}\n"
    f"Correct? {pred1 == example['answer']}"
)

&nbsp;
### F.3 使用验证器检查答案

- 本节没有代码（参见第 3 章）

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F03_raschka.webp" width="500px">

<br>
&nbsp;

### F.4 利用偏好和排行榜比较模型

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F04_raschka.webp" width="500px">

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F05_raschka.webp" width="500px">

- Elo 评分（"400 算法"）源自象棋排名：https://en.wikipedia.org/wiki/Performance_rating_(chess)
- 请注意，LM Arena 已改用统计的 Bradely-Terry 模型来给出类似 Elo 的分数，但依旧沿用成对排名的概念

In [ ]:
# 成对的“竞技场投票”，其中第一个模型是胜者，
# 第二个模型是败者
votes = [
    ("GPT-5", "Claude-3"),  # 第一组对决：GPT-5 优于 Claude-3
    ("GPT-5", "Llama-4"),
    ("Claude-3", "Llama-3"),
    ("Llama-4", "Llama-3"),
    ("Claude-3", "Llama-3"),
    ("GPT-5", "Llama-3"),
]

In [ ]:
def elo_ratings(vote_pairs, k_factor=32, initial_rating=1000):
    # 用相同的初始等级分初始化所有模型
    ratings = {
        model: initial_rating
        for pair in vote_pairs
        for model in pair
    }

    # 每场对局后更新等级分
    for winner, loser in vote_pairs:

        # 根据等级分计算当前胜者的预期得分
        expected_winner = 1.0 / (
            1.0 + 10 ** ((ratings[loser] - ratings[winner]) / 400.0)
        )

        # k_factor 决定等级分更新的敏感程度
        ratings[winner] = (
            ratings[winner] + k_factor * (1 - expected_winner)
        )
        ratings[loser] = (
            ratings[loser] + k_factor * (0 - (1 - expected_winner))
        )

    return ratings

In [ ]:
ratings = elo_ratings(votes, k_factor=32, initial_rating=1000)

for model in sorted(ratings, key=ratings.get, reverse=True):
    print(f"{model:8s} : {ratings[model]:.1f}")

- 期望胜者得分的计算如下：

$$\text{expected\_winner} \;=\; \frac{1}{1 + 10^{\tfrac{\text{rating\_loser} - \text{rating\_winner}}{400}}}
$$

- 直观理解：
    - 若 rating_winner >> rating_loser：
       - 指数 → 非常负
       - 分母 ≈ 1
       - expected_winner ≈ 1（几乎稳赢）
    - 若 rating_winner << rating_loser：
       - 指数 → 非常正
       - 分母 → 很大
       - expected_winner ≈ 0（几乎必败）
    - 若 rating_winner == rating_loser：
       - 指数 = 0
       - 分母 = 2
       - expected_winner = 0.5（势均力敌）

&nbsp;
### F.5 使用其他 LLM 评判回答

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F06_raschka.webp" width="500px">

- 本节通过另一个更大的 LLM 来自动评估微调后 LLM 的回答
- 我们使用的是 Open AI 的 200 亿参数指令微调版 gpt-oss 模型，可通过 ollama ([https://ollama.com](https://ollama.com)) 在本地运行

- Ollama 是一个高效运行 LLM 的开源应用
- 它封装了 llama.cpp ([https://github.com/ggerganov/llama.cpp](https://github.com/ggerganov/llama.cpp))，该项目用纯 C/C++ 实现 LLM 以最大化效率
- 请注意，它是用于 LLM 推理（文本生成）的工具，而不是用于训练或微调 LLM
- 运行下方代码前，请访问 [https://ollama.com](https://ollama.com) 并按照说明（例如单击 "Download" 按钮并下载与你的操作系统匹配的 ollama 应用）完成安装

- macOS 和 Windows 用户请打开刚下载的 ollama 应用，若提示安装命令行用法，请选择 "yes"
- Linux 用户可以使用 ollama 官网提供的安装命令
- 在电脑上运行 ollama 的方式有 3 种：

**1. `ollama serve`**

- 以服务器形式运行 ollama 后端，通常监听 `http://localhost:11434`。只有当我们通过 API 调用时才会加载模型；如果想在 Python 中调用 ollama，就需要这种方式。

**2. `ollama run gpt-oss:20b`**

- 这是一个便捷封装。如果服务器尚未运行，它会先启动，然后（首次）下载模型，并进入可与模型聊天的交互式终端。其背后仍然使用同一个服务器 API。

**3. Ollama 桌面应用**

- 自动运行相同的后端并提供 GUI（如上图所示）。
它还会应用默认设置（系统提示、温度、停止序列），因此输出可能与直接调用 API 不同。

---

**注意**:

- 按照上述方式在终端中运行 `ollama serve` 时，可能会遇到 `Error: listen tcp 127.0.0.1:11434: bind: address already in use` 报错
- 若遇到该问题，可尝试运行 `OLLAMA_HOST=127.0.0.1:11435 ollama serve`（如果该地址仍被占用，可依次递增端口直到找到空闲地址）

---

- 例如，若想体验 ollama，可以运行 `ollama run gpt-oss:20b` 试用 200 亿参数的 gpt-oss 20B 模型。该模型（约 13 GB）会在第一次运行此命令时自动下载。（或者，可以像前图那样在桌面应用中使用。）
  
```bash
ollama run gpt-oss:20b
```


- 输出类似如下：

```
$ ollama run gpt-oss:20b
pulling manifest 
pulling b112e727c6f1: 100% ▕█████████████████████████████████▏  13 GB                         
pulling fa6710a93d78: 100% ▕█████████████████████████████████▏ 7.2 KB                         
pulling f60356777647: 100% ▕█████████████████████████████████▏  11 KB                         
pulling d8ba2f9a17b3: 100% ▕█████████████████████████████████▏   18 B                         
pulling 55c108d8e936: 100% ▕█████████████████████████████████▏  489 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success
```

- 如需了解 gpt-oss 的更多细节，可阅读我的长文 [From GPT-2 to gpt-oss: Analyzing the Architectural Advances](https://magazine.sebastianraschka.com/p/from-gpt-2-to-gpt-oss-analyzing-the) 
- 使用 "gpt-oss:20b" 模型（20B 参数）需要 13 GB RAM；如果机器无法满足，可尝试更小的 `qwen3:4b` 模型（约需 4 GB RAM）
- 如果硬件允许，也可以改用更大的 120B gpt-oss (`qwen3:235b`) 甚至 235B 的 Qwen3 (`qwen3:235b`)
- 下载完成后，会看到一个可以与模型对话的命令行提示符
- 试着输入 "What is 1+2?"，模型会返回类似下述的输出

```
>>> What is 1+2?
Thinking...
User asks: "What is 1+2?" This is simple: answer 3. Provide explanation? Possibly ask for simple 
arithmetic. Provide answer: 3.
...done thinking.

1 + 2 = **3**
```


- 输入 `/bye` 可结束该会话

- 下方代码会在使用 ollama 评估上一节生成的测试集回答之前，先检查 ollama 会话是否正常运行

In [ ]:
import psutil

def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running

ollama_running = check_if_running("ollama")

if not ollama_running:
    raise RuntimeError(
        "Ollama not running. Launch ollama before proceeding."
    )
print("Ollama running:", check_if_running("ollama"))

- 除了之前使用的 `ollama run` 命令外，还可以通过其 REST API 在 Python 中调用模型，代码如下函数
- 在运行本笔记本后续的单元格前，请确保 ollama 仍在运行（前面的代码单元应打印 `"Ollama running: True"`）
- 接着运行下一个代码单元向模型发起查询

In [ ]:
import json
import requests


def query_model(
    prompt,
    model="gpt-oss:20b",
    # 如果通过 OLLAMA_HOST=127.0.0.1:11435 ollama serve 启动服务
    # 将地址中的端口从 11434 改为 11435
    url="http://localhost:11434/api/chat"
):
    # 以字典形式创建请求数据
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "options": {     # 要获得确定性回复，需要使用以下设置
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }

    # 发送 POST 请求
    with requests.post(url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()
        response_data = ""
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            response_json = json.loads(line)
            if "message" in response_json:
                response_data += response_json["message"]["content"]

    return response_data

In [ ]:
ollama_model = "gpt-oss:20b"
result = query_model("What is 1+2?", ollama_model)
print(result)

- 现在，可以使用上面定义的 `query_model` 函数来评估我们自己模型的回答

In [ ]:
def rubric_prompt(instruction, reference_answer, model_answer):
    rubric = (
        "You are a fair judge assistant. You will be given an instruction, "
        "a reference answer, and a candidate answer to evaluate, according "
        "to the following rubric:\n\n"
        "1: The response fails to address the instruction, providing "
        "irrelevant, incorrect, or excessively verbose content.\n"
        "2: The response partially addresses the instruction but contains "
        "major errors, omissions, or irrelevant details.\n"
        "3: The response addresses the instruction to some degree but is "
        "incomplete, partially correct, or unclear in places.\n"
        "4: The response mostly adheres to the instruction, with only "
        "minor errors, omissions, or lack of clarity.\n"
        "5: The response fully adheres to the instruction, providing a "
        "clear, accurate, and relevant answer in a concise and efficient "
        "manner.\n\n"
        "Now here is the instruction, the reference answer, and the "
        "response.\n"
    )

    prompt = (
        f"{rubric}\n"
        f"Instruction:\n{instruction}\n\n"
        f"Reference Answer:\n{reference_answer}\n\n"
        f"Answer:\n{model_answer}\n\n"
        f"Evaluation: "
    )
    return prompt

- `model_answer` 可以是我们自己模型生成的答案；此处为了简单起见，先硬编码了一个示例回答

In [ ]:
rendered_prompt = rubric_prompt(
    instruction=(
        "If all birds can fly, and a penguin is a bird, "
        "can a penguin fly?"
    ),
    reference_answer=(
        "Yes, according to the premise that all birds can fly, "
        "a penguin can fly."
    ),
    model_answer=(
        "Yes – under those premises a penguin would be able to fly."
    )
)
print(rendered_prompt)

In [ ]:
result = query_model(rendered_prompt, ollama_model)
print(result)